# 6. Exploitation de vos données structurées

## XPath : le langage pour naviguer dans votre base de données 

Les balises que vous venez d'ajouter au texte nous permet d'y naviguer. Le langage de navigation s'appele [XPath](https://fr.wikipedia.org/wiki/XPath). Sélectionnez une balise, notez l'adresse Xpath en haut du panneau d'édition, ouvrez le panneau `XPath`, et entrez l'adresse. Par exemple :

```
TEI/body/div/p/persName
```

S'il y a plus qu'un paragraphe, et une personne, vous allez découvrir que toutes les personnes sont mises en surbrillances et que vous pouvez sauter de l'une à l'autre. C'est cool, Word ne fait pas ça.

Si l'on veut être précis, on peut utiliser des **index**. Si l'on veut la deuxième personne dans le premier paragraphe :

```
TEI/body/div/p[1]/persName[2]
```

Si l'on veut la première personne dans _chaque paragraphe_ :

```
TEI/body/div/p/persName[1]
```

Et si l'on s'en fiche d'où se trouvent les personnes exactement dans l'arborescence :

```
TEI//persName
```

Exercises :

- [ ] Ciblez un élément balisé et essayez d'y naviguer seulement avec Xpath.

Il suffit de comprendre le principe. XPath est un langage informatique pour la machine, et l'IA peut vous aider à traduire, mais on ne sait pas quelle question formuler si l'on ignore la structure de votre base de données. 

## Quelques exemples pour vous motiver

La rélation entre questionement et balisage est normalement itérative : on se pose une question, on fait du balisage jusqu'à ce que l'on rencontre un inprévu, puis on raffine le questionement et la méthode de balisage et l'on continue. Les vrais résultats arrivent tout d'un coup, dans quelques secondes, à la fin d'un travail de quelques semaines. C'est par consequent un travail un peu « à l'aveugle » dans lequel il est facile à se perdre et à se décourager. Pour vous rassurer du potentiel final de ce travail, je vous présente quelques exemples d'analyses « entrée de game » que vous pouvez faire sur votre corpus.

Cette page s'exécute entièrement dans **votre navigateur** (JupyterLite, via GitHub Pages). Chaque participant a sa propre session : votre fichier XML reste sur votre ordinateur et n'est jamais envoyé sur GitHub.

Téléversez le texte que vous venez de baliser dans la cellule ci-dessous. Les scripts suivants l'analyseront dans votre session.

## Exemple 1 : Totaux

In [ ]:
# JupyterLite charge ces bibliothèques dans le navigateur si nécessaire.
%pip install -q beautifulsoup4 ipywidgets lxml

from pathlib import Path

import ipywidgets as widgets
from IPython.display import display

UPLOAD_DIR = Path("data")
UPLOAD_DIR.mkdir(exist_ok=True)
XML_PATH = None

upload = widgets.FileUpload(
    accept=".xml,application/xml,text/xml",
    multiple=False,
    description="Choisir un XML",
    button_style="primary",
)

status = widgets.Output()


def save_uploaded_file(change):
    global XML_PATH
    with status:
        status.clear_output()
        if not upload.value:
            return

        file_info = upload.value[0]
        content = file_info["content"]
        if hasattr(content, "tobytes"):
            content = content.tobytes()

        XML_PATH = UPLOAD_DIR / file_info["name"]
        XML_PATH.write_bytes(content)
        print(f"Fichier enregistré dans votre session : {XML_PATH}")
        print("Vous pouvez maintenant exécuter la cellule suivante.")


upload.observe(save_uploaded_file, names="value")
display(widgets.VBox([upload, status]))

In [ ]:
from pathlib import Path

from bs4 import BeautifulSoup

if "XML_PATH" not in globals() or XML_PATH is None or not XML_PATH.exists():
    upload_dir = Path("data")
    xml_paths = sorted(upload_dir.glob("*.xml"), key=lambda p: p.stat().st_mtime, reverse=True)
    if not xml_paths:
        raise FileNotFoundError(
            "Aucun fichier XML trouvé. Téléversez d'abord votre fichier dans la cellule ci-dessus."
        )
    XML_PATH = xml_paths[0]

print(f"Analyse de : {XML_PATH.name}")

results_dic = {}

with open(XML_PATH, "r", encoding="utf-8") as f:
    soup = BeautifulSoup(f.read(), "xml")

nodes = [node for p in soup.find_all("p") for node in p.find_all(True)]

for node in nodes:
    results_dic[node.tag] = results_dic.get(node.tag, 0) + 1

for tag, count in sorted(results_dic.items(), key=lambda item: item[1], reverse=True):
    print(f"{tag}: {count}")

## Ce que l'on apprend

Vous avez obtenu des listes, des comptes et des statistiques de paragraphes. Mais `Laetitia`, `Marina` ou `荊州` restent pour l'instant des chaînes de caractères. Deux formes peuvent renvoyer à la même entité, et une même forme peut renvoyer à plusieurs entités. C'est le problème de la désambiguïsation du chapitre suivant.

## Exercices

- [ ] Tester plusieurs expressions XPath dans l'éditeur.
- [ ] Filtrer `tagged_df` sur un type de balise.
- [ ] Trouver les formes les plus longues et les plus fréquentes.
- [ ] Comparer le nombre de balises par paragraphe.
- [ ] Écrire une question de recherche à laquelle ces résultats pourraient contribuer.